# HW02 — MLflow Experiment Tracking

In HW01, you built a versioned feature dataset for the Airbnb listing availability problem.

In this notebook, you will train several model versions and track them in MLflow.

The goal is not only to get a high score. The goal is to make every experiment reproducible:

- which dataset version was used
- which features were used
- which model was trained
- which parameters were used
- which metrics were produced
- which artifacts were saved
- which run should be considered the final candidate

MLflow server:

```text
http://185.50.38.163:33014
```

Use your assigned MLflow username/password and your assigned experiment name from the credentials sheet.

## Required output

By the end of this notebook, you must have:

1. At least **5 MLflow runs**.
2. At least **3 different experiment types**:
   - one intentionally leaky run
   - one baseline run
   - at least one clean real model
3. Logged parameters, metrics, tags, artifacts, and an sklearn Pipeline model.
4. A run comparison table.
5. One selected final candidate run.
6. A short explanation of why that run was selected.

Do not use future/label columns in your final clean model.

In [23]:
# If needed, install these in your local environment first:
# pip install pandas numpy scikit-learn matplotlib mlflow pyarrow

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

import mlflow
import mlflow.sklearn

RANDOM_STATE = 42

## 1. Configure MLflow

Fill in your assigned MLflow credentials.

Important:

- `MLFLOW_TRACKING_URI` is the shared MLflow server.
- `MLFLOW_USERNAME` and `MLFLOW_PASSWORD` are **not** your database credentials.
- `EXPERIMENT_NAME` must be your own assigned experiment name, for example:

```text
qbc12_hw02_student_nazanin_hesari
```

Do not use someone else's experiment name.

In [24]:
MLFLOW_TRACKING_URI = "http://185.50.38.163:33014"

# TODO: replace these with your assigned MLflow credentials.
MLFLOW_USERNAME = "student_mohammad_torkashvand"
MLFLOW_PASSWORD = "lRTc2sOa8vaNlsJWl0k"
EXPERIMENT_NAME = "qbc12_hw02_student_mohammad_torkashvand"

if MLFLOW_USERNAME == "student_your_username" or MLFLOW_PASSWORD == "your_mlflow_password":
    raise ValueError("Replace MLFLOW_USERNAME, MLFLOW_PASSWORD, and EXPERIMENT_NAME with your assigned values.")

os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else None)
print("Experiment ID:", experiment.experiment_id if experiment else None)

MLflow tracking URI: http://185.50.38.163:33014
Experiment: qbc12_hw02_student_mohammad_torkashvand
Experiment ID: 28


## 2. Load the HW01 dataset

Use the cleaned dataset produced by HW01.

Expected files:

```text
data/features/listing_availability_features_v1_audit_cleaned.csv
data/features/listing_availability_features_v1_audit_cleaned.parquet
data/features/listing_availability_features_v1_audit_cleaned_metadata.json
```

You may use CSV or Parquet. Parquet is preferred if available.

In [25]:
DATASET_VERSION = "v1_student"

FEATURE_DIR = Path("data/features")

parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"

# TODO: load the dataset.
# Prefer Parquet if it exists, otherwise use CSV.
if parquet_path.exists():
    feature_df = pd.read_parquet(parquet_path)
elif csv_path.exists():
    feature_df = pd.read_csv(csv_path)
else:
    raise NotImplementedError("Load feature_df from parquet_path or csv_path.")

# TODO: load metadata if metadata_path exists.
if metadata_path.exists():
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)

print(feature_df.shape)
feature_df.head()

(10480, 28)


,listing_id,room_type,property_type,accommodates,bedrooms,minimum_nights,maximum_nights,instant_bookable,bathrooms,is_superhost,...,avg_maximum_nights_calendar_last_90d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,sum_future_available_days_30d,avg_future_available_days_30d,high_demand_proxy,cutoff_date,version
0,27886,Private room,Private room in houseboat,0.066667,0.058824,0.002,0.315836,False,1.5,True,...,30.0,11,0.354839,3.000000,30.0,1,0.033333,1,2025-12-10,v1
1,28871,Private room,Private room in rental unit,0.066667,0.058824,0.001,0.648577,False,1,True,...,730.0,9,0.290323,1.870968,730.0,4,0.133333,1,2025-12-10,v1
2,29051,Private room,Private room in condo,0.066667,0.058824,0.001,0.648577,False,1,True,...,730.0,12,0.387097,2.000000,730.0,0,0.000000,1,2025-12-10,v1
3,44391,Entire home/apt,Entire rental unit,0.200000,0.117647,0.002,0.648577,False,1.5,False,...,730.0,0,0.000000,3.000000,730.0,0,0.000000,1,2025-12-10,v1
4,48373,Entire home/apt,Entire rental unit,0.200000,0.117647,0.002,1.000000,False,1.5,False,...,1125.0,0,0.000000,3.000000,1125.0,0,0.000000,1,2025-12-10,v1


## 3. Define target and forbidden columns

The target is:

```text
high_demand_proxy
```

The following columns must **not** be used as clean model inputs:

```text
listing_id
cutoff_date
dataset_version
future_calendar_days_observed_30d
future_available_days_30d
future_available_rate_30d
high_demand_proxy
```

Why?

- `high_demand_proxy` is the label.
- `future_*` columns are from the label window.
- `listing_id`, `cutoff_date`, and `dataset_version` are audit/entity fields, not predictive features.

You will intentionally use one future column in the **leaky run only** to show what leakage looks like. Your final model must be clean.

In [26]:
feature_df.columns = ['listing_id', 'room_type', 'property_type', 'accommodates', 'bedrooms',
       'minimum_nights', 'maximum_nights', 'instant_bookable', 'bathrooms',
       'is_superhost', 'count', 'name', 'total_reviews_before_cutoff',
       'unique_reviewers_before_cutoff', 'days_since_last_review',
       'available_days_last_90d', 'available_rate_last_90d',
       'avg_minimum_nights_calendar_last_90d',
       'avg_maximum_nights_calendar_last_90d', 'future_available_days_30d',
       'future_available_rate_30d', 'avg_minimum_nights_calendar_last_30d',
       'avg_maximum_nights_calendar_last_30d', 'sum_future_available_days_30d',
       'avg_future_available_days_30d', 'high_demand_proxy', 'cutoff_date',
       'dataset_version']

In [27]:
TARGET_COL = "high_demand_proxy"

FORBIDDEN_MODEL_COLUMNS = [
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "sum_future_available_days_30d",
    "avg_future_available_days_30d",
    "high_demand_proxy",
]

# TODO: check that TARGET_COL exists.
if TARGET_COL not in feature_df.columns.to_list():
    raise KeyError('The target column does not exist in your dataframe')

try:
    # TODO: create y.
    y = feature_df[TARGET_COL]

    # TODO: create clean feature list by excluding FORBIDDEN_MODEL_COLUMNS.
    clean_feature_cols = [x for x in feature_df.columns.to_list() if x not in FORBIDDEN_MODEL_COLUMNS]

    # TODO: create X_clean.
    X = feature_df[clean_feature_cols]
except:
    raise NotImplementedError("Create y, clean_feature_cols, and X_clean.")

print("Target distribution:")
print(y.value_counts(normalize=True).sort_index())

print("Clean feature count:", len(clean_feature_cols))
print(clean_feature_cols)

Target distribution:
high_demand_proxy
0    0.362786
1    0.637214
Name: proportion, dtype: float64
Clean feature count: 20
['room_type', 'property_type', 'accommodates', 'bedrooms', 'minimum_nights', 'maximum_nights', 'instant_bookable', 'bathrooms', 'is_superhost', 'count', 'name', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff', 'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d', 'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d', 'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d']


## 4. Create one intentionally leaky feature set

This run is supposed to be wrong.

Create `X_leaky` by allowing `future_available_rate_30d` into the features.

The point is to show that a model can look excellent for the wrong reason. Log this run with:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
```

Do not select this run as your final model.

In [28]:
LEAKAGE_COLUMN = "future_available_rate_30d"

# TODO: create leaky_feature_cols.
# It should include the clean features plus LEAKAGE_COLUMN.
# It must still exclude the target itself.
leaky_feature_cols = clean_feature_cols + [LEAKAGE_COLUMN]


print("Leaky feature count:", len(leaky_feature_cols))
print("Leakage column included:", LEAKAGE_COLUMN in leaky_feature_cols)

Leaky feature count: 21
Leakage column included: True


## 5. Train/test split

Use a stratified split.

Why stratified?

The target is not perfectly balanced, so the train and test sets should preserve the class ratio.

In [29]:
# TODO: split X_clean and y.
# Use test_size=0.20, random_state=42, stratify=y.
X_leaky = X.copy()
X_leaky[LEAKAGE_COLUMN] = feature_df[LEAKAGE_COLUMN]

try:
    X_train, X_test,  y_train, y_test = train_test_split(X, y, test_size=0.2, random_state= 42, stratify= y)
except:
    raise NotImplementedError("Create X_train, X_test, y_train, y_test.")

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

Train shape: (8384, 20)
Test shape: (2096, 20)
Train target rate: 0.6371660305343512
Test target rate: 0.6374045801526718


## 6. Build preprocessing

Use an sklearn `ColumnTransformer`.

Required preprocessing:

- numeric columns:
  - median imputation
  - standard scaling
- categorical columns:
  - most-frequent imputation
  - one-hot encoding

The logged model must be a full sklearn `Pipeline`, not just the estimator.

In [30]:
def make_one_hot_encoder():
    """Return OneHotEncoder compatible with multiple sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

X['bathrooms'] = X['bathrooms'].astype(float)
X['is_superhost'] = X['is_superhost'].astype(bool)

# TODO: identify numeric_cols and categorical_cols from X_clean.
# Hint: numeric columns usually have dtype int/float.
# Everything else can be treated as categorical.
try:
    numeric_cols = X.dtypes[(X.dtypes == float) | (X.dtypes == int)].index.to_list()
    categorical_cols = X.dtypes[~((X.dtypes == float) | (X.dtypes == int))].index.to_list()
except:
    raise NotImplementedError("Create numeric_cols and categorical_cols.")

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_cols),
        ("categorical", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

Numeric columns: 15
Categorical columns: 5


C:\Users\moham\AppData\Local\Temp\ipykernel_7672\1268411187.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['bathrooms'] = X['bathrooms'].astype(float)
C:\Users\moham\AppData\Local\Temp\ipykernel_7672\1268411187.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['is_superhost'] = X['is_superhost'].astype(bool)


## 7. Evaluation helpers

Complete the evaluation helper.

Every run must log the same metric set:

```text
accuracy
precision
recall
f1
roc_auc
```

Use `zero_division=0` for precision/recall/f1.

In [31]:
def get_positive_scores(model, X):
    """Return positive-class scores for binary classifiers."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    return model.predict(X)


def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5):
    """Evaluate a fitted binary classifier."""
    try:
        # TODO:
        # 1. get positive scores
        y_score = get_positive_scores(model, X_test)

        # 2. convert scores to predictions using threshold
        y_pred = (y_score > threshold).astype(int)

        # 3. calculate accuracy, precision, recall, f1, roc_auc
        metrics = {
        accuracy : accuracy_score(y_test, y_pred),
        precision : precision_score(y_test, y_pred),
        recall : recall_score(y_test, y_pred),
        f1 : f1_score(y_test, y_pred),
        roc_auc : roc_auc_score(y_test, y_pred)
        }

        # 4. return metrics dict, y_pred, y_score
        return metrics, y_pred, y_score
    except:
        raise NotImplementedError("Complete evaluate_binary_classifier.")

## 8. Artifact helpers

Each serious run should save useful artifacts:

- confusion matrix image
- classification report JSON
- feature column list JSON
- dataset metadata snapshot JSON

Artifacts are important because MLflow should store more than scalar metrics.

In [32]:
ARTIFACT_DIR = Path("outputs/mlflow_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def save_run_artifacts(run_name, y_true, y_pred, feature_cols, metadata):
    """Save local artifact files for one run and return the run artifact directory."""
    # TODO:
    try:
        # 1. create a run-specific artifact folder
        run_dir = ARTIFACT_DIR/run_name
        (run_dir).mkdir(parents= True, exist_ok= True)

        # 2. save confusion_matrix.png
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot()
        plt.title(f"Confusion Matrix — {run_name}")
        plt.savefig(run_dir / "confusion_matrix.png")
        plt.close() 

        # 3. save classification_report.json
        classification_report = classification_report(y_true, y_pred, output_dict= True)
        with open(run_dir / "classification_report.json", "w") as f:
            json.dump(classification_report, f, indent=2)
        
        # 4. save feature_columns.json
        with open(run_dir/"feature_columns.json", "w") as f:
            json.dump(feature_cols, f, indent= 2)

        # 5. save dataset_metadata_snapshot.json
        with open(run_dir/"dataset_metadata_snapshot.json", "w") as f:
            json.dump(metadata, f, indent= 2)
    except:
        raise NotImplementedError("Complete save_run_artifacts.")

## 9. MLflow run helper

Complete a helper that:

1. fits the pipeline,
2. evaluates it,
3. logs params,
4. logs metrics,
5. logs tags,
6. logs artifacts,
7. logs the full sklearn Pipeline model.

Use the same helper for all model versions. That is the point of experiment tracking.

In [33]:
def run_mlflow_experiment(
    run_name,
    pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    model_params,
    tags,
    threshold=0.5,
):
    # TODO: implement this function.
    # Required MLflow calls:
    # - mlflow.start_run(run_name=run_name)
    # - mlflow.log_params(...)
    # - mlflow.log_metrics(...)
    # - mlflow.set_tags(...)
    # - mlflow.log_artifacts(...)
    # - mlflow.sklearn.log_model(...)
    with mlflow.start_run(run_name= run_name):
        # Training and prediction process
        pipeline.fit(X_train, y_train)
        y_pred = (pipeline.predict_proba(X_test)[:, 1] > threshold).astype(int)

        # Calculation of metrics
        acc = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # set tags and params and ...
        mlflow.log_params(model_params)

        mlflow.log_metric("accuracy score", acc)
        mlflow.log_metric("precision score", precision)
        mlflow.log_metric("f1 score", f1)

        mlflow.set_tags(tags)

        mlflow.log_artifacts("outputs/mlflow_artifacts/")

        mlflow.sklearn.log_model(sk_model=intentionally_pipeline, artifact_path="model")

## 10. Run 0 — intentionally leaky model

This run is wrong on purpose.

Use a real model, but include `future_available_rate_30d`.

Expected behavior: performance may look suspiciously strong.

Required tags:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
model_family = logistic_regression
```

In [34]:
# TODO:
# 1. split X_leaky and y using the same stratified split settings
# 2. build a LogisticRegression pipeline
# 3. log the run to MLflow
X_leaky_train, X_leaky_test,  y_leaky_train, y_leaky_test = train_test_split(X_leaky, y, test_size=0.2, random_state= 42, stratify= y)
intentionally_pipeline = Pipeline(
    steps=[
        ("prepocesser", preprocessor),
        ("model", LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", max_iter=1000))
    ]
)

model_params = {
    "C": 1.0,         
    "penalty": "l2",    
    "solver": "lbfgs",
    "max_iter": 1000
}
run_mlflow_experiment(
    "intentionally_leaky",
    intentionally_pipeline,
    X_leaky_train,
    X_leaky_test,
    y_leaky_train,
    y_leaky_test,
    X_leaky.columns.to_list(),
    model_params,
    {"leakage_status":"leaky", "known_defect": "uses future_available_rate_30d", "model_family":"logistic_regression"},
    threshold=0.5,
)

2026/06/16 16:46:19 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run intentionally_leaky at: http://185.50.38.163:33014/#/experiments/28/runs/374cf9f7546a4e7b9dadfec9d639e39b
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/28


## 11. Run 1 — dummy baseline

Train a `DummyClassifier(strategy="most_frequent")`.

This tells you what a useless model can achieve.

If your real model barely beats this, your model is weak.

In [35]:
dummy_pipeline = Pipeline(steps=[
    ("model", DummyClassifier(strategy="most_frequent"))
])

dummy_params = {"strategy": "most_frequent"}

run_mlflow_experiment(
    "dummy_baseline",
    dummy_pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    X_train.columns.tolist(),
    dummy_params,
    {"model_family": "dummy", "leakage_status": "clean"},
    threshold=0.5
)

2026/06/16 16:46:27 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run dummy_baseline at: http://185.50.38.163:33014/#/experiments/28/runs/151d030ee9bf4006abac6832478abd09
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/28


## 12. Run 2 — clean logistic regression

Train your first clean real model.

Use only `X_clean`.

Required tags:

```text
leakage_status = clean
model_family = logistic_regression
```

In [36]:
clean_pipeline = Pipeline(
    steps=[
        ("prepocesser", preprocessor),
        ("model", LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", max_iter=1000))
    ]
)

model_params = {
    "C": 1.0,        
    "penalty": "l2", 
    "solver": "lbfgs",
    "max_iter": 1000
}
run_mlflow_experiment(
    "clean logistic regression",
    clean_pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    X.columns.to_list(),
    model_params,
    {"leakage_status":"clean", "model_family":"logistic_regression"},
    threshold=0.5,
)

2026/06/16 16:46:37 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run clean logistic regression at: http://185.50.38.163:33014/#/experiments/28/runs/87203dcb44054b1da92daea1cc23a6bc
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/28


## 13. Run 3 — class-weighted logistic regression

Train logistic regression with:

```python
class_weight="balanced"
```

Compare precision and recall against the previous clean logistic model.

In [37]:
class_weighted_pipeline = Pipeline(
    steps=[
        ("prepocesser", preprocessor),
        ("model", LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", max_iter=1000, class_weight="balanced"))
    ]
)

model_params = {
    "C": 1.0,        
    "penalty": "l2", 
    "solver": "lbfgs",
    "max_iter": 1000,
    "class_weight": "balanced"
}
run_mlflow_experiment(
    "class weighted logistic regression",
    class_weighted_pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    X.columns.to_list(),
    model_params,
    {"leakage_status":"clean", "model_family":"logistic_regression", "class_weight":"balanced"},
    threshold=0.5,
)

2026/06/16 16:46:45 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run class weighted logistic regression at: http://185.50.38.163:33014/#/experiments/28/runs/2b67ac2449e147d9a6fd374c1d8d6445
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/28


## 14. Run 4 — threshold tuning

Use a fitted probability model and test several decision thresholds.

Suggested thresholds:

```text
0.30, 0.40, 0.50, 0.60
```

You may log one run per threshold.

The goal is to see how precision/recall/f1 change when the threshold changes.

In [38]:
model_params = {
    "C": 1.0,        
    "penalty": "l2", 
    "solver": "lbfgs",
    "max_iter": 1000,
    "class_weight": "balanced"
}

for thr in [0.3, 0.4, 0.5, 0.6]:
    run_mlflow_experiment(
        "class weighted logistic regression for threshold tuning",
        class_weighted_pipeline,
        X_train,
        X_test,
        y_train,
        y_test,
        X.columns.to_list(),
        model_params,
        {"leakage_status":"clean", "model_family":"logistic_regression", "class_weight":"balanced", "threshold":thr},
        threshold=thr,
    )

2026/06/16 16:46:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run class weighted logistic regression for threshold tuning at: http://185.50.38.163:33014/#/experiments/28/runs/ccc3aa5c47f04775b7a76d67db5db0c5
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/28


2026/06/16 16:47:03 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run class weighted logistic regression for threshold tuning at: http://185.50.38.163:33014/#/experiments/28/runs/ea5e902bb237447c8b762ded78cd539b
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/28


2026/06/16 16:47:12 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run class weighted logistic regression for threshold tuning at: http://185.50.38.163:33014/#/experiments/28/runs/f68c6a0f5b47461cb28e832092d70cd8
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/28


2026/06/16 16:47:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run class weighted logistic regression for threshold tuning at: http://185.50.38.163:33014/#/experiments/28/runs/c9bb6916c3ce4ff3ae5489bf8a77778c
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/28


## 15. Run 5 — tree-based model

Train a `RandomForestClassifier`.

This compares a nonlinear model against logistic regression.

Log at least these parameters:

```text
n_estimators
max_depth
min_samples_leaf
class_weight
random_state
```

In [39]:
tree_based_pipeline = Pipeline(
    steps=[
        ("prepocesser", preprocessor),
        ("model", RandomForestClassifier())
    ]
)

model_params = RandomForestClassifier().get_params()

run_mlflow_experiment(
    "tree based model",
    tree_based_pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    X.columns.to_list(),
    model_params,
    {"leakage_status":"clean", "model_family":"logistic_regression", "class_weight":"balanced"},
    threshold=0.5,
)

2026/06/16 16:47:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run tree based model at: http://185.50.38.163:33014/#/experiments/28/runs/0eaaef4a513f4d75b6d23c4f34946e55
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/28


## 16. Compare MLflow runs

Use `mlflow.search_runs` to retrieve your experiment runs.

Compare at least:

```text
run name
leakage status
model family
accuracy
precision
recall
f1
roc_auc
```

Do not select a leaky run as final candidate.

In [40]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1 DESC"]
)

# use only columns that exist
comparison_df = runs[[
    "tags.mlflow.runName",
    "tags.leakage_status",
    "tags.model_family",
    "metrics.accuracy score",
    "metrics.precision score",
    "metrics.f1 score"
]].copy()

comparison_df.columns = [
    "run_name",
    "leakage_status",
    "model_family",
    "accuracy",
    "precision",
    "f1"
]

metric_cols = ["accuracy", "precision", "f1"]
comparison_df[metric_cols] = comparison_df[metric_cols].round(4)

# show clean runs only at the top
comparison_df = comparison_df.sort_values(
    ["leakage_status", "f1"],
    ascending=[True, False]
)

comparison_df

,run_name,leakage_status,model_family,accuracy,precision,f1
0,tree based model,clean,logistic_regression,0.8507,0.8709,0.8847
9,tree based model,clean,logistic_regression,0.8488,0.8705,0.8831
6,clean logistic regression,clean,logistic_regression,0.8430,0.8538,0.8808
15,clean logistic regression,clean,logistic_regression,0.8430,0.8538,0.8808
18,tree based model,clean,logistic_regression,0.8445,0.8665,0.8799
3,class weighted logistic regression for thresho...,clean,logistic_regression,0.8421,0.8592,0.8790
12,class weighted logistic regression for thresho...,clean,logistic_regression,0.8421,0.8592,0.8790
4,class weighted logistic regression for thresho...,clean,logistic_regression,0.8383,0.8426,0.8785
13,class weighted logistic regression for thresho...,clean,logistic_regression,0.8383,0.8426,0.8785
2,class weighted logistic regression for thresho...,clean,logistic_regression,0.8435,0.8711,0.8782


## 17. Select final candidate

Pick the best **clean** run.

Do not choose the leaky run.

Selection should be based on:

- f1
- roc_auc
- precision/recall tradeoff
- no leakage
- full preprocessing Pipeline logged

Write a short explanation.

In [41]:
# TODO: set BEST_RUN_ID to the selected clean run ID.
# We sorted the results with f1 score then chose the index that has the biggest value for f1
BEST_RUN_ID = runs['run_id'][comparison_df['f1'].idxmax()]

if BEST_RUN_ID is None:
    raise ValueError("Set BEST_RUN_ID to your selected clean MLflow run ID.")

client.set_tag(BEST_RUN_ID, "selected_for_serving", "true")
client.set_tag(BEST_RUN_ID, "production_candidate", "true")

print("Selected best run:", BEST_RUN_ID)

Selected best run: 0eaaef4a513f4d75b6d23c4f34946e55


## Final explanation

Write 3–6 sentences:

- Which run did you select?
- Why did you select it?
- Why did you reject the leaky run?
- What would you try next?

In [42]:
# TODO: replace this text.
final_explanation = """
At first I tried different situations to train the model for example we used logistic regression and random forest as our base models
to predict the results.
As we can see from metrics dummy and leaky output is not as good as we expect, because in the first one we just used a very simple 
rule to predict result and in the second one leaky feature will make the model related to the output that it is not a proper way
finally we can see all runs with their id and all metrics and parammeters. even all models exists in the mlflow server that finally we selected
one of them that you can find its run id above.
"""

print(final_explanation)


At first I tried different situations to train the model for example we used logistic regression and random forest as our base models
to predict the results.
As we can see from metrics dummy and leaky output is not as good as we expect, because in the first one we just used a very simple 
rule to predict result and in the second one leaky feature will make the model related to the output that it is not a proper way
finally we can see all runs with their id and all metrics and parammeters. even all models exists in the mlflow server that finally we selected
one of them that you can find its run id above.

